In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

np.random.seed(2612)

In [2]:
# Configurações de cores e estilos
colors = {
    "AGD": "orangered"
}

estilo_setas = {
    'width': 0.003,
    'headwidth': 2,
    'headlength': 3
}

estilo_segmentos_aux = {
    'linestyle': 'dashed',
    'width': 0.003
}

In [3]:
shift = np.array([128., -56.])

def f(x):
    return (x[0] - shift[0])**2 + (x[1] - shift[1])**2 + (x[0] - shift[0])*(x[1] - shift[1])

def grad_f(x):
    return np.array([2*(x[0] - shift[0]) + x[1] - shift[1], 2*(x[1] - shift[1]) + x[0] - shift[0]])

a = 1
b = 1
c = 1.5

def phi(x):
    return a * x[0]**2 + b * x[1]**2 + c * x[0] * x[1]

def grad_phi(x):
    return np.array([2 * a * x[0] + c * x[1], 2 * b * x[1] + c * x[0]])

def inv_grad_phi(y):
    denom = 4 * a * b - c**2
    return np.array([(2 * b * y[0] - c * y[1])/denom, (2 * a * y[1] - c * y[0])/denom])

In [4]:
# Parâmetros fixos
k_start = 2
number_of_iterations = 100
xopt = np.array([0., 0.]) + shift

# Pontos iniciais
theta0 = np.array([0., 0.])
x0 = inv_grad_phi(theta0)
y0 = x0

# Diferentes tamanhos de passo para testar
eta_values = [0.0025, 0.01, 0.04, 0.16]
T = 5  # Tempo final para a EDO

# EDO para AGD
def AGD_ode(state, t):
    x, y, vx, vy = state
    dxdt = vx
    dydt = vy
    # Evitar divisão por zero para t muito pequeno
    if t < 1e-3:
        damping = 0
    else:
        damping = 3/t
    dvxdt = -damping * vx - grad_f([x, y])[0]
    dvydt = -damping * vy - grad_f([x, y])[1]
    return [dxdt, dydt, dvxdt, dvydt]

In [5]:
# PRIMEIRA PASSADA: Coletar todos os pontos para determinar os limites comuns
print("Coletando dados para determinar limites comuns...")
all_points = []

for eta in eta_values:
    # Reinicializar para cada eta
    k = k_start
    AGD_x_points = [x0]
    AGD_y_points = [y0]
    
    # Primeira iteração
    x1 = y0 - eta * grad_f(y0)
    AGD_x_points.append(x1)
    
    # Iterações do AGD
    for i in range(number_of_iterations):
        mu_k = (k-1)/(k+2)
        
        y1 = AGD_x_points[-1] + mu_k * (AGD_x_points[-1] - AGD_x_points[-2])
        AGD_y_points.append(y1)
        x1 = y1 - eta * grad_f(y1)
        AGD_x_points.append(x1)
        
        k += 1
    
    AGD_x_points = np.array(AGD_x_points)
    AGD_y_points = np.array(AGD_y_points)
    
    # Tempo para integração da EDO
    t_ode = np.linspace(0.1, T, 1000)
    
    # Condição inicial para a EDO - estimar velocidade inicial
    vx0 = (AGD_x_points[1,0] - AGD_x_points[0,0]) / eta
    vy0 = (AGD_x_points[1,1] - AGD_x_points[0,1]) / eta
    initial_state = [x0[0], x0[1], vx0, vy0]
    
    # Solução da EDO para AGD
    sol_primal_agd = odeint(AGD_ode, initial_state, t_ode)
    AGD_ode_x = sol_primal_agd[:, 0]
    AGD_ode_y = sol_primal_agd[:, 1]
    
    # Coletar todos os pontos
    all_points.extend(AGD_x_points)
    all_points.extend(AGD_y_points)
    all_points.extend(np.column_stack([AGD_ode_x, AGD_ode_y]))

# Converter para array e calcular limites comuns
all_points = np.array(all_points)
x_coords = all_points[:, 0]
y_coords = all_points[:, 1]

# Calcular limites comuns com padding
x_range_common = (x_coords.min() - 0.1, x_coords.max() + 0.1)
y_range_common = (y_coords.min() - 0.1, y_coords.max() + 0.1)

print(f"Limites comuns: x={x_range_common}, y={y_range_common}")

# SEGUNDA PASSADA: Gerar os gráficos com limites comuns
for eta in eta_values:
    print(f"Processando eta = {eta}")
    
    # Reinicializar para cada eta
    k = k_start
    AGD_x_points = [x0]
    AGD_y_points = [y0]
    
    # Primeira iteração
    x1 = y0 - eta * grad_f(y0)
    AGD_x_points.append(x1)
    
    # Iterações do AGD
    for i in range(number_of_iterations):
        mu_k = (k-1)/(k+2)
        
        y1 = AGD_x_points[-1] + mu_k * (AGD_x_points[-1] - AGD_x_points[-2])
        AGD_y_points.append(y1)
        x1 = y1 - eta * grad_f(y1)
        AGD_x_points.append(x1)
        
        k += 1
    
    AGD_x_points = np.array(AGD_x_points)
    AGD_y_points = np.array(AGD_y_points)
    
    # Tempo para integração da EDO
    t_ode = np.linspace(0.1, T, 1000)
    
    # Condição inicial para a EDO - estimar velocidade inicial
    vx0 = (AGD_x_points[1,0] - AGD_x_points[0,0]) / eta
    vy0 = (AGD_x_points[1,1] - AGD_x_points[0,1]) / eta
    initial_state = [x0[0], x0[1], vx0, vy0]
    
    # Solução da EDO para AGD
    sol_primal_agd = odeint(AGD_ode, initial_state, t_ode)
    AGD_ode_x = sol_primal_agd[:, 0]
    AGD_ode_y = sol_primal_agd[:, 1]
    
    # Criar figura
    fig, ax_primal = plt.subplots(figsize=(10, 10))
    
    # Grade para contornos (usando os limites comuns)
    xgrid = np.linspace(x_range_common[0], x_range_common[1], 100)
    ygrid = np.linspace(y_range_common[0], y_range_common[1], 100)
    X, Y = np.meshgrid(xgrid, ygrid)
    Z_f_primal = f([X, Y])
    
    # Plot dos contornos
    ax_primal.contour(X, Y, Z_f_primal, levels=20, cmap='inferno', alpha=0.7)
    
    # Plot dos pontos importantes
    x_0_plot = ax_primal.scatter(x0[0], x0[1], color='k', s=150, label='ponto inicial', zorder=5)
    x_opt_plot = ax_primal.scatter(xopt[0], xopt[1], color='gold', s=250, marker='*', 
                                  edgecolors='red', label='ponto ótimo', zorder=5)
    
    # Plot dos pontos do AGD discreto
    AGD_pontos_plot = ax_primal.scatter(AGD_x_points[:,0], AGD_x_points[:,1], marker="D", 
                                       color=colors['AGD'], s=80, label=f'AGD (η={eta})', zorder=4)
    
    # Plot da solução da EDO
    AGD_ode_plot, = ax_primal.plot(AGD_ode_x, AGD_ode_y, 'r-', linewidth=2.5, 
                                  label='AGD (EDO contínua)', zorder=3)
    
    # Setas para AGD discreto
    AGD_arrows_x0_to_x1 = AGD_x_points[1:] - AGD_x_points[0:-1]
    if len(AGD_arrows_x0_to_x1) > 0:
        ax_primal.quiver(AGD_x_points[0:-1,0], AGD_x_points[0:-1,1], 
                        AGD_arrows_x0_to_x1[:,0], AGD_arrows_x0_to_x1[:,1], 
                        angles='xy', scale_units='xy', color=colors['AGD'], 
                        width=estilo_setas['width'], 
                        headwidth=estilo_setas['headwidth'], 
                        headlength=estilo_setas['headlength'], 
                        scale=1, zorder=3, alpha=0.7)
    
    # Configurações do gráfico - USANDO LIMITES COMUNS
    ax_primal.set_xlim(x_range_common)
    ax_primal.set_ylim(y_range_common)
    ax_primal.set_title(f'AGD - Espaço Primal (η={eta}, T={T})')
    ax_primal.set_xlabel('x')
    ax_primal.set_ylabel('y')
    ax_primal.set_aspect('equal')
    ax_primal.grid(True, alpha=0.3)
    
    # Legenda
    handles_primal = [x_0_plot, x_opt_plot, AGD_pontos_plot, AGD_ode_plot]
    labels_primal = [h.get_label() for h in handles_primal]
    ax_primal.legend(handles_primal, labels_primal, loc='upper right')
    
    # Salvar figura
    nome_base = f'AGD_eta_{eta:.3f}_T_{T}'
    caminho_arquivo = f"imagens_dissertação/comparação_EDO_e_método_AGD/{nome_base}.pdf"
    fig.savefig(caminho_arquivo, bbox_inches="tight", dpi=300)
    plt.close(fig)  # Fechar figura para liberar memória
    
    print(f"Figura salva: {caminho_arquivo}")

print("Processamento concluído!")

Coletando dados para determinar limites comuns...
Limites comuns: x=(-0.1, 137.65425064137975), y=(-69.49743218493072, 3.3000000000000003)
Processando eta = 0.0025
Figura salva: imagens_dissertação/comparação_EDO_e_método_AGD/AGD_eta_0.003_T_5.pdf
Processando eta = 0.01
Figura salva: imagens_dissertação/comparação_EDO_e_método_AGD/AGD_eta_0.010_T_5.pdf
Processando eta = 0.04
Figura salva: imagens_dissertação/comparação_EDO_e_método_AGD/AGD_eta_0.040_T_5.pdf
Processando eta = 0.16
Figura salva: imagens_dissertação/comparação_EDO_e_método_AGD/AGD_eta_0.160_T_5.pdf
Processamento concluído!
